# Session 2 — Build an Agentic RAG System

In this session you will build, step by step, an agent that answers
questions about NVIDIA's financial results using the company's own filings.

The pipeline has four stages:

1. **Chunk** 
2. **Vectorize** 
3. **Search** 
4. **Assemble an agent** 

The corpus is in `data/02_processed/`: NVIDIA's 10-K filings for fiscal 2025 and 2026,
trimmed to their substantive pages, plus the transcripts of the two matching earnings
calls.

Before starting, take a look at the documents themselves in `data/01_raw/` and
`data/02_processed/`. That is the content everything here relies on.

## Setup

The helpers used in this notebook are given to you in `utils/build.py`. Open it and have a
look before going further.

In [ ]:
import json
from pathlib import Path

from utils.build import build_chunk_records, load_document, save_chunks

PROCESSED_DIR = Path("../data/02_processed")
CHUNKS_DIR = Path("../data/03_chunks")
ALL_CHUNKS_PATH = CHUNKS_DIR / "all_chunks.json"

sorted(p.name for p in PROCESSED_DIR.iterdir() if p.suffix in {".pdf", ".txt"})

## Exercise 1 — Chunking the documents

The first step of a RAG system is cutting the corpus into small pieces. How much a model can
read at once depends on the model, and the longer the context, the more it loses track of what
matters. Two parameters set the trade-off:

- **`chunk_size`**: maximum characters in a chunk. Too large wastes tokens and buries the
  answer among irrelevant text; too small truncates the information.
- **`overlap`**: characters shared by consecutive chunks. Without it, a sentence cut by a
  boundary is whole in neither.

With `chunk_size=30` and `overlap=8`, each chunk starts 22 characters after the previous
one and repeats its last 8:

![Chunks of 30 characters overlapping by 8](assets/chunking-overlap.png)

**Objective.** Implement the chunking logic: splitting a document's text into overlapping
pieces.

**Your task.** Complete `chunk_string` below, respecting its signature and the intent
documented in its docstring.

**Hint.** Use [`CharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/CharacterTextSplitter)
from `langchain_text_splitters`. 

In [ ]:
def chunk_string(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into overlapping, non-blank chunks.

    Parameters
    ----------
    text : str
        The text to split.
    chunk_size : int
        Maximum number of characters in a chunk.
    overlap : int
        Number of characters each chunk shares with the previous one.

    Returns
    -------
    list[str]
        The chunks, in order. Blank chunks are dropped.
    """
    # YOUR CODE HERE

### Check your implementation

In [ ]:
chunks = chunk_string("abcde, fghij. klmn\n opqrs!", chunk_size=10, overlap=3)

assert chunks == ["abcde, fgh", "fghij. klm", "klmn\n opqr", "pqrs!"]
print("OK:", len(chunks), "chunks")

### Chunk every document

The next step is to orchestrate chunking across a list of documents. It proceeds as follows:

1. `load_document(path)` returns the full text of a document along with the fiscal year it
   covers.
2. `chunk_string`, the function you have just written, cuts that text into overlapping pieces.
3. `build_chunk_records(text_chunks, document_name, year)` gives each chunk an `id` and records
   which document and year it came from.
4. `save_chunks(records, path)` writes those records to a JSON file, one per document, in
   `data/03_chunks/`.

In [ ]:
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

for document_path in sorted(PROCESSED_DIR.iterdir()):
    if document_path.suffix not in {".pdf", ".txt"}:
        continue  # skip anything that is not one of our documents

    text, year = load_document(str(document_path))
    text_chunks = chunk_string(text, chunk_size=1000, overlap=200)
    records = build_chunk_records(text_chunks, document_path.name, year)

    save_chunks(records, str(CHUNKS_DIR / f"{document_path.stem}.json"))
    print(f"{document_path.name}: {len(records)} chunks")

### Gather them into one index

Retrieval runs against a single index covering the whole corpus. The cell below reads the
per-document files back and concatenates them into `all_chunks.json`, which every later step
works from.

In [ ]:
all_chunks = []

for document_chunks_path in sorted(CHUNKS_DIR.glob("*.json")):
    if document_chunks_path != ALL_CHUNKS_PATH:  # do not read the gathered file into itself
        all_chunks.extend(json.loads(document_chunks_path.read_text()))

save_chunks(all_chunks, str(ALL_CHUNKS_PATH))
print(f"{len(all_chunks)} chunks in {ALL_CHUNKS_PATH}")

## Exercise 2 — Vectorizing the chunks

A question rarely uses the same words as the passage that answers it. For example, a filing
writes "revenue" where the question says "sales". 

To tackle this problem, an embedding model maps a text to a vector of
numbers, positioned so that texts carrying a similar meaning land close together. Comparing
two vectors then tells you how close two passages are in meaning.

The model below is a small one, which runs locally on your CPU. The first run downloads it
(90 MB).

The vectors it produces are normalised, which is what makes two of them comparable.

In [ ]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from numpy.typing import NDArray

from utils.build import save_vectors

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
    show_progress=True,
)

print("vector dimension:", len(embedding_model.embed_query("test")))

**Objective.** Turn a list of texts into the matrix holding one vector per text.

**Your task.** Complete `embed_texts` below, respecting its signature and the intent
documented in its docstring.

**Hint.** [`embed_documents`](https://reference.langchain.com/python/langchain-huggingface/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings)
embeds a whole list of texts in one call and returns a list of lists. `np.asarray` turns that
into a matrix.

In [ ]:
def embed_texts(texts: list[str], model: HuggingFaceEmbeddings) -> NDArray[np.float32]:
    """Turn each text into a vector.

    Parameters
    ----------
    texts : list[str]
        The texts to embed.
    model : HuggingFaceEmbeddings
        The embedding model, already loaded.

    Returns
    -------
    NDArray[np.float32]
        One row per text, of shape (len(texts), embedding dimension).
    """
    # YOUR CODE HERE

### Check your implementation

In [ ]:
examples = ["revenue grew sharply", "sales increased a lot", "the cafeteria serves lunch"]
vectors = embed_texts(examples, embedding_model)

assert vectors.shape == (3, 384)
np.testing.assert_allclose(
    vectors[:, :3],
    [[0.031227, -0.031101, 0.004742], [-0.034080, -0.028074, 0.002028], [-0.046444, 0.116769, 0.036383]],
    atol=1e-5,
)
print("OK:", vectors.shape[0], "vectors of dimension", vectors.shape[1])

### Vectorize the whole corpus

The cell below embeds every chunk of `all_chunks.json` and writes the result to
`data/04_vectors/`. `save_vectors` stores the matrix together with the chunk ids it came from,
so that row `i` can always be traced back to the chunk it encodes.

In [ ]:
%%time
VECTORS_PATH = Path("../data/04_vectors/chunk_vectors.npz")
VECTORS_PATH.parent.mkdir(parents=True, exist_ok=True)

all_chunks = json.loads(ALL_CHUNKS_PATH.read_text())
vectors = embed_texts([chunk["text"] for chunk in all_chunks], embedding_model)

save_vectors(vectors, [chunk["id"] for chunk in all_chunks], str(VECTORS_PATH))
print(f"{vectors.shape[0]} vectors of dimension {vectors.shape[1]} in {VECTORS_PATH}")